# NB2 — Fault Injection

Injects synthetic sensor faults into the full WADI dataset and performs the train/test split, producing a three-class output: **normal (0)**, **attack (1)**, **fault (2)**. Key steps:

- **Select** injectable sensors (CLR-only subset, 17 total)
- **Calibrate** per-sensor fault magnitudes from normal data statistics
- **Fit** causal propagation models along documented control-loop edges
- **Inject** faults randomly across the full continuous stream
- **Split** stratified by fault type to guarantee evaluation coverage
- **Save** `data/wadi_faulted.parquet`

**Why CLR sensors only?** Faults on closed-loop control sensors propagate through the controller to downstream valves, pumps, and process variables — producing realistic multi-sensor disturbance signatures. Indication-only sensors produce isolated single-sensor anomalies that are a fundamentally different detection problem outside this study's scope.

**Prerequisites:** Run NB1 first.

In [1]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)

DATA_DIR    = Path("data")
FIGURES_DIR = DATA_DIR / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED        = 42
TARGET_FAULT_PCT   = 0.20
WINDOW_SIZE        = 30    # seconds per window for train/test split
TEST_RATIO         = 0.20
# Rows, not seconds. WaDi: ~15.7 rows/unique-timestamp.
# 600 rows ≈ 38 real seconds; 1800 rows ≈ 115 real seconds.
FAULT_DURATION_MIN = 600
FAULT_DURATION_MAX = 1800
MIN_COVERAGE_RATIO = 0.10
BINARY_MAX_UNIQUE  = 5

np.random.seed(RANDOM_SEED)

catalog_v2 = yaml.safe_load(Path("fault_catalog.yaml").read_text())
SENSOR_DEFS       = catalog_v2["sensors"]
PROP_EDGES        = catalog_v2["allowed_propagation_edges"]
CONSEQUENCE_ONLY  = set(catalog_v2["primary_injection_policy"]["consequence_only_tags"])

df = pd.read_parquet(DATA_DIR / "wadi_prepared.parquet")
sensor_cols_ref = json.loads((DATA_DIR / "sensor_cols.json").read_text())
SENSOR_COLS = sensor_cols_ref["sensor_cols"]

print(f"Loaded: {df.shape}")
print(f"Sensor columns: {len(SENSOR_COLS)}")
print(f"Catalog sensor definitions: {len(SENSOR_DEFS)}")
print(f"Propagation edges: {len(PROP_EDGES)}")

Loaded: (957374, 100)
Sensor columns: 98
Catalog sensor definitions: 38
Propagation edges: 13


## 1. Primary Injection Policy — Eligible Sensors

Filters the full sensor list down to 17 injectable sensors in three stages:

1. **Remove binary/sparse sensors** — fewer than 5 unique values or <10% data coverage
2. **Apply catalog policy** — exclude tags marked `inject: false`, CO/SP/SPEED/FQ suffixes, and consequence-only tags
3. **Restrict to CLR sensors** — keep only `closed_loop_response` profile sensors (control-loop participants)

Three SP setpoint sensors (`2_FIC_101/201/401_SP`) are added back for **drift-only** injection — CLR-adjacent with good IQR/std ratios, representing realistic slow reference drift.

In [2]:
ALLOWED_TAG_SUFFIXES = (
    "_LT_", "_FIT_", "_FIC_", "_PIT_", "_DPIT_", "_AIT_", "_PIC_"
)

df_normal = df[df["label"] == 0].copy()

eligible   = []
ineligible = []
for col in SENSOR_COLS:
    series = df_normal[col].dropna()
    if series.nunique() <= BINARY_MAX_UNIQUE:
        ineligible.append((col, "binary"))
        continue
    if len(series) / len(df_normal) < MIN_COVERAGE_RATIO:
        ineligible.append((col, "sparse"))
        continue
    eligible.append(col)

# Step 1: Apply v2 primary injection policy (measurement tags, no CO/SP/SPEED/FQ)
policy_injectable = []
skipped = []
for col in eligible:
    if SENSOR_DEFS.get(col, {}).get("inject", True) is False:
        skipped.append((col, "inject:false"))
        continue
    if col in CONSEQUENCE_ONLY:
        skipped.append((col, "consequence_only"))
        continue
    if any(col.endswith(s) or f"_{s.strip('_')}_" in col for s in ("_CO", "_SP")):
        skipped.append((col, "forbidden_suffix"))
        continue
    if "_SPEED" in col or "_FQ_" in col:
        skipped.append((col, "forbidden_suffix"))
        continue
    if not any(t in col for t in ALLOWED_TAG_SUFFIXES):
        skipped.append((col, "not_measurement_type"))
        continue
    policy_injectable.append(col)

# Step 2: Restrict to closed-loop response sensors only.
injectable = [
    col for col in policy_injectable
    if SENSOR_DEFS.get(col, {}).get("default_profile", "measurement_only") == "closed_loop_response"
]
excluded_indication_only = [
    col for col in policy_injectable
    if SENSOR_DEFS.get(col, {}).get("default_profile", "measurement_only") == "measurement_only"
]

# Step 3: Add FIC setpoint sensors for drift-only injection.
# Setpoints have good IQR/std ratios (~1.4–1.7) unlike CLR PV sensors
# (e.g. 2_PIC_003_PV has IQR/std = 0.043). Drift on a setpoint sensor is
# a realistic fault: slow unauthorized modification or controller reference drift.
# These are CLR-adjacent — part of the same control loops as the FIC_PV sensors.
CLR_SP_DRIFT_SENSORS = [s for s in ["2_FIC_101_SP", "2_FIC_201_SP", "2_FIC_401_SP"]
                        if s in SENSOR_COLS]
injectable = injectable + CLR_SP_DRIFT_SENSORS

injectable_set = set(injectable)

print(f"Eligible sensors:               {len(eligible)}")
print(f"After policy filter:            {len(policy_injectable)}")
print(f"Indication-only (excluded):     {len(excluded_indication_only)}")
print(f"CLR injectable (PV sensors):    {len(injectable) - len(CLR_SP_DRIFT_SENSORS)}")
print(f"CLR-adjacent SP (drift-only):   {len(CLR_SP_DRIFT_SENSORS)}  {CLR_SP_DRIFT_SENSORS}")
print(f"Total injectable:               {len(injectable)}")
print(f"\nAll injectable sensors:")
for col in injectable:
    tag = "SP-drift-only" if col in CLR_SP_DRIFT_SENSORS else SENSOR_DEFS.get(col, {}).get("default_profile", "?")
    print(f"  {col:40s} [{tag}]")

Eligible sensors:               67
After policy filter:            36
Indication-only (excluded):     22
CLR injectable (PV sensors):    14
CLR-adjacent SP (drift-only):   3  ['2_FIC_101_SP', '2_FIC_201_SP', '2_FIC_401_SP']
Total injectable:               17

All injectable sensors:
  1_FIT_001_PV                             [closed_loop_response]
  1_LT_001_PV                              [closed_loop_response]
  2_FIC_101_PV                             [closed_loop_response]
  2_FIC_201_PV                             [closed_loop_response]
  2_FIC_301_PV                             [closed_loop_response]
  2_FIC_401_PV                             [closed_loop_response]
  2_FIC_501_PV                             [closed_loop_response]
  2_FIC_601_PV                             [closed_loop_response]
  2_LT_001_PV                              [closed_loop_response]
  2_LT_002_PV                              [closed_loop_response]
  2_PIC_003_PV                             [closed_loop_

## 2. Per-Sensor Magnitude Calibration

Computes per-sensor statistics from **all normal data** to anchor fault magnitudes to each sensor's natural operating range:

- **IQR** (Q75 − Q25) — primary magnitude scale for bias, noise, and dropout modes
- **Std** — used for `monotonic_drift`, where IQR-based scaling fails on bimodal sensors (e.g., `2_PIC_003_PV` has IQR/std = 0.043, so 4 × IQR ≈ 0.17σ — invisible against normal variation)
- **Mean, Q25, Q75** — retained for downstream reference

Statistics are computed before splitting — the difference from using training data alone is negligible at ~950K normal rows.

In [3]:
df_normal_train = df[df["label"] == 0]

sensor_iqr  = {}
sensor_std  = {}
sensor_q25  = {}
sensor_q75  = {}
sensor_mean = {}

for col in injectable:
    s = df_normal_train[col].dropna()
    q25 = float(s.quantile(0.25))
    q75 = float(s.quantile(0.75))
    iqr = q75 - q25
    sensor_iqr[col]  = max(iqr, 1e-6)   # guard against zero-IQR sensors
    sensor_std[col]  = max(float(s.std()), 1e-6)
    sensor_q25[col]  = q25
    sensor_q75[col]  = q75
    sensor_mean[col] = float(s.mean())

print("Per-sensor IQR calibration (first 20 injectable sensors):")
print(f"{'Sensor':40s}  {'IQR':>10}  {'Std':>10}  {'Mean':>10}")
for col in injectable[:20]:
    print(f"  {col:38s}  {sensor_iqr[col]:10.4f}  {sensor_std[col]:10.4f}  {sensor_mean[col]:10.4f}")

Per-sensor IQR calibration (first 20 injectable sensors):
Sensor                                           IQR         Std        Mean
  1_FIT_001_PV                                1.8499      0.8468      0.5140
  1_LT_001_PV                                13.9864      8.7962     56.2960
  2_FIC_101_PV                                0.0947      0.1450      0.1173
  2_FIC_201_PV                                0.0813      0.1200      0.1117
  2_FIC_301_PV                                0.0719      0.1161      0.0996
  2_FIC_401_PV                                0.1058      0.1359      0.1179
  2_FIC_501_PV                                0.0892      0.1358      0.1109
  2_FIC_601_PV                                0.0770      0.1234      0.0998
  2_LT_001_PV                                 0.9737      0.6291     69.7148
  2_LT_002_PV                                 5.8043      3.7262     75.3337
  2_PIC_003_PV                                0.0180      0.4202      0.2116
  2_PIT_002_PV    

## 3. Causal Propagation Models

For each documented edge in `fault_catalog.yaml`, a data-driven model is fitted from normal training data:

- **Lag** — cross-correlation argmax over lags 0–500 rows (~32 s), finding the true delay between upstream and downstream sensors
- **Coefficient** — downstream response amplitude: `r × (σ_downstream / σ_upstream)`

During injection, a root fault propagates along CLR edges as:
```
downstream[lag:] += coeff × root_perturbation[:-lag]
```
Only `closed_loop_response` sensors participate in propagation. `measurement_only` sensors are never propagated to.

In [4]:
MAX_LAG = 500   # search up to 500 rows (~32 real seconds) for optimal lag

# Build propagation model: from_tag → [(to_tag, lag, coeff, profile)]
propagation_models: dict[str, list[dict]] = {}

# Use the longest continuous normal-training block for cross-correlation
df_nt = df_normal_train.sort_values("timestamp").reset_index(drop=True)

for edge in PROP_EDGES:
    from_tag = edge["from"]
    profile  = edge["profile"]

    if from_tag not in df_nt.columns:
        continue

    from_series = df_nt[from_tag].fillna(method="ffill").values.astype(float)

    for to_tag in edge["to"]:
        if to_tag not in df_nt.columns:
            continue
        if to_tag not in injectable_set and to_tag not in set(SENSOR_COLS):
            continue

        to_series = df_nt[to_tag].fillna(method="ffill").values.astype(float)

        # Cross-correlation: corr(from[:-lag], to[lag:]) for lag in 0..MAX_LAG
        n = len(from_series)
        lags = range(0, min(MAX_LAG + 1, n // 4))
        cc_vals = []
        for lag in lags:
            if lag == 0:
                a, b = from_series, to_series
            else:
                a, b = from_series[:-lag], to_series[lag:]
            if len(a) < 100:
                cc_vals.append(0.0)
                continue
            # Pearson r
            a_c = a - a.mean()
            b_c = b - b.mean()
            denom = (np.std(a_c) * np.std(b_c))
            r = float(np.mean(a_c * b_c) / denom) if denom > 1e-10 else 0.0
            cc_vals.append(r)

        best_lag = int(np.argmax(np.abs(cc_vals)))
        best_r   = cc_vals[best_lag]

        # Amplitude coefficient: downstream_std / upstream_std * r
        # (how many units of downstream change per unit of upstream change)
        up_std   = sensor_std.get(from_tag, 1.0)
        down_std = sensor_std.get(to_tag, sensor_std.get(to_tag, 1.0))
        if to_tag not in sensor_std:
            s = df_nt[to_tag].dropna()
            down_std = max(float(s.std()), 1e-6)
        coeff = best_r * (down_std / up_std)

        propagation_models.setdefault(from_tag, []).append({
            "to":      to_tag,
            "lag":     best_lag,
            "coeff":   coeff,
            "r":       best_r,
            "profile": profile,
        })

print("Fitted propagation models:")
for from_tag, edges in propagation_models.items():
    for e in edges:
        print(f"  {from_tag:25s} → {e['to']:25s}  lag={e['lag']:4d} rows  r={e['r']:+.3f}  coeff={e['coeff']:+.4f}  [{e['profile']}]")

/tmp/ipykernel_159652/2914123062.py:16: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  from_series = df_nt[from_tag].fillna(method="ffill").values.astype(float)
/tmp/ipykernel_159652/2914123062.py:24: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  to_series = df_nt[to_tag].fillna(method="ffill").values.astype(float)


Fitted propagation models:
  1_LT_001_PV               → 1_FIT_001_PV               lag=   0 rows  r=-0.131  coeff=-0.0126  [closed_loop_response]
  1_LT_001_PV               → 2_LT_001_PV                lag= 396 rows  r=-0.019  coeff=-0.0014  [closed_loop_response]
  1_LT_001_PV               → 2_LT_002_PV                lag=  17 rows  r=-0.013  coeff=-0.0055  [closed_loop_response]
  1_FIT_001_PV              → 1_AIT_001_PV               lag=   0 rows  r=+0.062  coeff=+1.0205  [closed_loop_response]
  1_FIT_001_PV              → 1_AIT_002_PV               lag=   0 rows  r=+0.242  coeff=+0.0193  [closed_loop_response]
  1_FIT_001_PV              → 1_AIT_003_PV               lag=   0 rows  r=+0.040  coeff=+0.0089  [closed_loop_response]
  1_FIT_001_PV              → 1_AIT_004_PV               lag=   0 rows  r=-0.379  coeff=-10.9046  [closed_loop_response]
  1_FIT_001_PV              → 1_AIT_005_PV               lag=   0 rows  r=-0.510  coeff=-0.0301  [closed_loop_response]
  2_LT_001_P

## 4. Fault Applier Functions

Six fault modes, each targeting a distinct failure mechanism:

| Mode | Fault type label | Description |
|---|---|---|
| `bias_step` | bias | Additive step offset (1–3 × IQR) |
| `monotonic_drift` | drift | Linear trend + de-trended noise (2–4 × σ) |
| `increased_noise` | precision_degradation | Added Gaussian noise (1–3 × σ) |
| `frozen_last_value` | stuck_at | Freezes at the first value in the fault window |
| `burst_dropout` | intermittent_dropout | Single contiguous NaN block (20–60% of window) |
| `sparse_dropout` | intermittent_dropout | Scattered NaN dropout at low rate (5–25%) |

Fault modes are sampled **randomly** per sensor event. The train/test split (Section 8) stratifies by fault type to guarantee evaluation coverage without constraining the injection itself.

In [5]:
def apply_monotonic_drift(series: pd.Series, target_std_multiple: float,
                          std: float) -> pd.Series:
    """Monotonic linear drift + de-trended Brownian noise, std-calibrated.

    Magnitude = target_std_multiple × std, guaranteeing 2–4σ displacement
    regardless of IQR/std ratio. IQR-calibration fails for sensors like
    2_PIC_003_PV (IQR/std = 0.043) where 4×IQR = only 0.17σ — invisible
    against a 3.4σ normal envelope.
    """
    s     = series.copy().astype("float32")
    n     = len(s)
    total = target_std_multiple * std * np.random.choice([-1, 1])
    trend = np.linspace(0, total, n)
    raw   = np.cumsum(np.random.normal(0, abs(total) * 0.15 / max(n ** 0.5, 1), n))
    noise = raw - np.linspace(raw[0], raw[-1], n)
    return (s + trend + noise.astype("float32")).astype("float32")


def apply_bias_step(series: pd.Series, offset_iqr_multiple: float,
                    iqr: float) -> pd.Series:
    """Additive step offset = offset_iqr_multiple * iqr."""
    s      = series.copy().astype("float32")
    offset = float(offset_iqr_multiple * iqr * np.random.choice([-1, 1]))
    return (s + offset).astype("float32")


def apply_increased_noise(series: pd.Series, noise_factor: float,
                           std: float) -> pd.Series:
    """Add Gaussian noise with std = noise_factor * training_std."""
    s     = series.copy().astype("float32")
    noise = np.random.normal(0, noise_factor * std, len(s)).astype("float32")
    return (s + noise).astype("float32")


def apply_frozen_last_value(series: pd.Series) -> pd.Series:
    """Freeze at the first value in the fault window (last-valid-reading model)."""
    frozen_val = float(series.iloc[0]) if not np.isnan(series.iloc[0]) else float(series.dropna().iloc[0])
    return pd.Series(frozen_val, index=series.index, dtype="float32")


def apply_burst_dropout(series: pd.Series, burst_fraction: float) -> pd.Series:
    """One contiguous NaN block of burst_fraction * len(series) rows."""
    s = series.copy().astype("float32")
    n = len(s)
    burst_len = max(1, int(burst_fraction * n))
    start_idx = int(np.random.randint(0, max(1, n - burst_len)))
    s.iloc[start_idx : start_idx + burst_len] = np.nan
    return s


def apply_sparse_dropout(series: pd.Series, dropout_rate: float) -> pd.Series:
    """Scattered NaN at low dropout_rate."""
    s    = series.copy().astype("float32")
    mask = np.random.random(len(s)) < dropout_rate
    s[mask] = np.nan
    return s


# monotonic_drift uses std-based magnitude (2–4σ) — guarantees visible displacement
# on all sensor types regardless of IQR/std ratio.
FAULT_PARAM_RANGES_V2 = {
    "bias_step":         {"offset_iqr_multiple": (1.0, 3.0)},
    "monotonic_drift":   {"target_std_multiple": (2.0, 4.0)},
    "increased_noise":   {"noise_factor": (1.0, 3.0)},
    "frozen_last_value": {},
    "burst_dropout":     {"burst_fraction": (0.20, 0.60)},
    "sparse_dropout":    {"dropout_rate": (0.05, 0.25)},
}

FAULT_MODE_TO_LABEL = {
    "bias_step":         "bias",
    "monotonic_drift":   "drift",
    "increased_noise":   "precision_degradation",
    "frozen_last_value": "stuck_at",
    "burst_dropout":     "intermittent_dropout",
    "sparse_dropout":    "intermittent_dropout",
}

CLR_SP_DRIFT = {"2_FIC_101_SP", "2_FIC_201_SP", "2_FIC_401_SP"}

# PIC/PIT pressure sensors with bimodal distributions (IQR/std = 0.043):
# normal operation already spans the ±4σ range, so magnitude-based faults
# are indistinguishable from normal state transitions.
# Restricted to dropout modes only.
DROPOUT_ONLY_SENSORS = {"2_PIC_003_PV", "2_PIT_003_PV"}


def sample_fault_mode(sensor: str, rng: np.random.Generator) -> str:
    """Sample a random fault mode for a sensor.
    - SP setpoints: drift only
    - PIC/PIT bimodal sensors: dropout only
    - All other CLR PV sensors: all modes
    """
    if sensor in CLR_SP_DRIFT:
        return "monotonic_drift"
    modes = SENSOR_DEFS.get(sensor, {}).get("allowed_fault_modes", list(FAULT_PARAM_RANGES_V2.keys()))
    available = [m for m in modes if m in FAULT_PARAM_RANGES_V2]
    if sensor in DROPOUT_ONLY_SENSORS:
        dropout_modes = [m for m in available if "dropout" in m]
        if dropout_modes:
            available = dropout_modes
    if not available:
        available = list(FAULT_PARAM_RANGES_V2.keys())
    return str(rng.choice(available))


def apply_fault_v2(series: pd.Series, fault_mode: str, param: float,
                   iqr: float, std: float) -> pd.Series:
    """Dispatch fault mode to applier."""
    if fault_mode == "bias_step":
        return apply_bias_step(series, param, iqr)
    elif fault_mode == "monotonic_drift":
        return apply_monotonic_drift(series, param, std)
    elif fault_mode == "increased_noise":
        return apply_increased_noise(series, param, std)
    elif fault_mode == "frozen_last_value":
        return apply_frozen_last_value(series)
    elif fault_mode == "burst_dropout":
        return apply_burst_dropout(series, param)
    elif fault_mode == "sparse_dropout":
        return apply_sparse_dropout(series, param)
    return series


print("Fault applier functions defined.")

Fault applier functions defined.


## 5. Injection Function

Applies faults in two tiers:

- **Tier 1 — root fault:** injects the chosen fault mode directly into the target sensor window
- **Tier 2 — causal propagation:** for CLR sensors, propagates the perturbation to documented downstream sensors using fitted lag and coefficient

Non-overlapping windows are enforced by tracking used timestamps globally across all sensors.

In [6]:
def inject_faults_rigorous(
    df_split_normal: pd.DataFrame,
    injectable_sensors: list[str],
    propagation_models: dict[str, list[dict]],
    sensor_iqr: dict[str, float],
    sensor_std: dict[str, float],
    catalog_v2: dict,
    target_fault_pct: float,
    duration_min: int,
    duration_max: int,
    rng: np.random.Generator,
) -> pd.DataFrame:

    df = df_split_normal.sort_values("timestamp").reset_index(drop=True)
    sensor_defs = catalog_v2["sensors"]

    target_fault_rows = int(len(df) * target_fault_pct)
    avg_duration      = (duration_min + duration_max) / 2
    faults_per_sensor = int(target_fault_rows / (len(injectable_sensors) * avg_duration))
    faults_per_sensor = max(faults_per_sensor, 0)

    if faults_per_sensor == 0:
        n_use = max(1, int(target_fault_rows / avg_duration))
        injectable_sensors = list(rng.choice(
            injectable_sensors, size=min(n_use, len(injectable_sensors)), replace=False
        ))
        faults_per_sensor = 1

    fault_rows       = []
    global_used_ts: set = set()

    for sensor in injectable_sensors:
        sensor_data = df[df[sensor].notna()].copy()
        if len(sensor_data) < duration_max * 2:
            continue

        n       = len(sensor_data)
        profile = sensor_defs.get(sensor, {}).get("default_profile", "measurement_only")
        iqr     = sensor_iqr.get(sensor, 1.0)
        std     = sensor_std.get(sensor, 1.0)

        prop_edges = propagation_models.get(sensor, []) if profile == "closed_loop_response" else []

        for _ in range(faults_per_sensor):
            fault_mode = sample_fault_mode(sensor, rng)
            cfg        = FAULT_PARAM_RANGES_V2[fault_mode]
            duration   = int(rng.integers(duration_min, duration_max + 1))

            placed = False
            for _ in range(50):
                start_idx = int(rng.integers(0, max(1, n - duration)))
                end_idx   = start_idx + duration
                window_ts = set(sensor_data.iloc[start_idx:end_idx]["timestamp"])
                if window_ts.isdisjoint(global_used_ts):
                    global_used_ts.update(window_ts)
                    placed = True
                    break

            if not placed:
                continue

            window = sensor_data.iloc[start_idx:end_idx].copy()

            if cfg:
                param_key = list(cfg.keys())[0]
                param = float(rng.uniform(*cfg[param_key]))
            else:
                param = 0.0

            original_root     = window[sensor].copy()
            window[sensor]    = apply_fault_v2(window[sensor], fault_mode, param, iqr, std)
            root_perturbation = (window[sensor] - original_root).values.astype(float)

            affected = [sensor]

            # Downstream perturbation = coeff * root_perturbation, delayed by lag rows.
            for edge in prop_edges:
                to_tag = edge["to"]
                if to_tag not in window.columns or to_tag in affected:
                    continue

                lag   = edge["lag"]
                coeff = edge["coeff"]
                w_len = len(window)

                if lag >= w_len:
                    continue

                ds_perturb = np.zeros(w_len, dtype=float)
                effective  = root_perturbation[:w_len - lag]
                ds_perturb[lag:lag + len(effective)] = coeff * effective

                window[to_tag] = (window[to_tag].values.astype(float) + ds_perturb).astype("float32")
                affected.append(to_tag)

            fault_label = FAULT_MODE_TO_LABEL.get(fault_mode, fault_mode)

            window["label"]                  = 2
            window["fault_type"]             = fault_label
            window["fault_sensor"]           = sensor
            window["fault_start"]            = window["timestamp"].iloc[0]
            window["fault_end"]              = window["timestamp"].iloc[-1]
            window["fault_severity"]         = round(float(param), 4)
            window["fault_sensor_count"]     = len(affected)
            window["fault_affected_sensors"] = ",".join(affected)
            window["fault_mode"]             = fault_mode
            window["fault_profile"]          = profile
            fault_rows.append(window)

    if not fault_rows:
        return pd.DataFrame()
    return pd.concat(fault_rows, ignore_index=True)


print("inject_faults_rigorous defined.")

inject_faults_rigorous defined.


## 6. Run Injection

Injects faults randomly across the full normal dataset (~10 min). All 17 injectable sensors receive random fault mode sampling — no pre-assignment. The stratified split in Section 8 ensures all fault types appear in both train and test.

In [7]:
rng = np.random.default_rng(RANDOM_SEED)
df_all_normal = df[df["label"] == 0].copy()
print(f"Injecting faults ({len(df_all_normal):,} normal rows, target {TARGET_FAULT_PCT:.0%})...")

fault_df = inject_faults_rigorous(
    df_split_normal    = df_all_normal,
    injectable_sensors = injectable,
    propagation_models = propagation_models,
    sensor_iqr         = sensor_iqr,
    sensor_std         = sensor_std,
    catalog_v2         = catalog_v2,
    target_fault_pct   = TARGET_FAULT_PCT,
    duration_min       = FAULT_DURATION_MIN,
    duration_max       = FAULT_DURATION_MAX,
    rng                = rng,
)

if fault_df.empty:
    print("WARNING: No faults injected!")
else:
    n_events    = fault_df.groupby(["fault_sensor", "fault_start"]).ngroups
    multi_count = (fault_df["fault_sensor_count"] > 1).sum()
    clr_rows    = (fault_df["fault_profile"] == "closed_loop_response").sum()
    print(f"Fault events:          {n_events}")
    print(f"Fault rows:            {len(fault_df):,}  ({len(fault_df)/len(df_all_normal)*100:.1f}%)")
    print(f"Multi-sensor rows:     {multi_count:,}  ({multi_count/len(fault_df)*100:.1f}%)")
    print(f"CLR (propagated) rows: {clr_rows:,}  ({clr_rows/len(fault_df)*100:.1f}%)")
    print(f"Avg sensors/event:     {fault_df['fault_sensor_count'].mean():.1f}")
    print(f"\nFault mode breakdown:")
    for fm, cnt in fault_df["fault_mode"].value_counts().items():
        label = FAULT_MODE_TO_LABEL.get(fm, fm)
        print(f"  {fm:28s} ({label:25s}): {cnt:,}")

Injecting faults (947,397 normal rows, target 20%)...
Fault events:          153
Fault rows:            186,622  (19.7%)
Multi-sensor rows:     140,163  (75.1%)
CLR (propagated) rows: 153,265  (82.1%)
Avg sensors/event:     2.8

Fault mode breakdown:
  monotonic_drift              (drift                    ): 47,263
  sparse_dropout               (intermittent_dropout     ): 36,902
  bias_step                    (bias                     ): 32,393
  burst_dropout                (intermittent_dropout     ): 31,219
  increased_noise              (precision_degradation    ): 19,462
  frozen_last_value            (stuck_at                 ): 19,383


## 7. Combine

Merges faulted rows back into the full dataset in two steps:

1. **Remove** normal rows at fault timestamps — replaces them with the injected fault version at full row density
2. **Concatenate** cleaned dataset + all fault rows and sort by timestamp

**Why no deduplication?** The injection function tracks used timestamps via `global_used_ts`, so no two events share timestamps. Each fault window retains the original ~15.7 rows/timestamp density of the WaDi data. Deduplicating to 1 row/timestamp would collapse fault regions to 6% of their intended density, destroying the rolling-feature signal for `stuck_at` (which relies on rolling std approaching 0 over a constant-value window).

In [8]:
for col in ["fault_type", "fault_sensor", "fault_start", "fault_end",
            "fault_severity", "fault_sensor_count", "fault_affected_sensors",
            "fault_mode", "fault_profile"]:
    if col not in df.columns:
        df[col] = np.nan

fault_ts  = set(fault_df["timestamp"].astype(str))
drop_mask = (df["label"] == 0) & (df["timestamp"].astype(str).isin(fault_ts))
df_clean  = df[~drop_mask].reset_index(drop=True)
print(f"Normal rows removed at fault timestamps: {drop_mask.sum():,}")
print(f"Fault rows to merge:                     {len(fault_df):,}")

df_injected = pd.concat([df_clean, fault_df], ignore_index=True)
df_injected = df_injected.sort_values("timestamp").reset_index(drop=True)

print(f"\nInjected dataset shape: {df_injected.shape}")
print(f"\nLabel counts:")
for lv, ln in [(0, "normal"), (1, "attack"), (2, "fault")]:
    print(f"  {ln} ({lv}): {(df_injected['label'] == lv).sum():>9,}")

fault_mask_  = df_injected["label"] == 2
avg_sensors  = df_injected.loc[fault_mask_, "fault_sensor_count"].mean()
multi_pct    = (df_injected.loc[fault_mask_, "fault_sensor_count"] > 1).mean() * 100
print(f"\nFault rows — avg sensors affected: {avg_sensors:.2f}")
print(f"Fault rows — multi-sensor (%):     {multi_pct:.1f}%")
print(f"\nFault type distribution:")
print(df_injected[df_injected["label"] == 2]["fault_type"].value_counts().to_string())

Normal rows removed at fault timestamps: 189,597
Fault rows to merge:                     186,622


/tmp/ipykernel_159652/1133993617.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_injected = pd.concat([df_clean, fault_df], ignore_index=True)



Injected dataset shape: (954399, 109)

Label counts:
  normal (0):   757,800
  attack (1):     9,977
  fault (2):   186,622

Fault rows — avg sensors affected: 2.75
Fault rows — multi-sensor (%):     75.1%

Fault type distribution:
fault_type
intermittent_dropout     68121
drift                    47263
bias                     32393
precision_degradation    19462
stuck_at                 19383


## 8. Stratified Train/Test Split

Assigns each row to train or test (80/20) while keeping fault and attack events intact:

- **Fault events** — each unique `(fault_sensor, fault_start)` event is treated as an atomic unit; stratified by **fault type** to guarantee all types appear in both splits
- **Normal + attack rows** — grouped into 30-second timestamp windows, stratified by label (normal vs. attack)

This mirrors real dataset collection: faults are injected into the continuous stream, then the recording is split for ML evaluation.

In [9]:
rng_split = np.random.default_rng(RANDOM_SEED)
df_injected["split"] = None

# --- Fault events: stratify by fault_type ---
fault_mask   = df_injected["label"] == 2
fault_events = (df_injected[fault_mask][["fault_sensor", "fault_start", "fault_type"]]
                .drop_duplicates())

print("Fault event stratification:")
for ft in sorted(fault_events["fault_type"].unique()):
    ft_events = fault_events[fault_events["fault_type"] == ft].copy()
    keys      = list(zip(ft_events["fault_sensor"].tolist(), ft_events["fault_start"].tolist()))
    perm      = rng_split.permutation(len(keys))
    shuffled  = [keys[i] for i in perm]
    n_test    = max(1, int(len(shuffled) * TEST_RATIO))
    for i, (sensor, start) in enumerate(shuffled):
        split_val = "test" if i < n_test else "train"
        m = fault_mask & (df_injected["fault_sensor"] == sensor) & (df_injected["fault_start"] == start)
        df_injected.loc[m, "split"] = split_val
    print(f"  {ft:30s}: {len(keys):3d} events → test={n_test}, train={len(keys)-n_test}")

# --- Normal + attack rows: 30s timestamp windows stratified by label ---
non_fault_mask = ~fault_mask
min_ts = df_injected["timestamp"].min()
df_injected.loc[non_fault_mask, "_wid"] = (
    (df_injected.loc[non_fault_mask, "timestamp"] - min_ts)
    .dt.total_seconds() // WINDOW_SIZE
)

windows = df_injected.loc[non_fault_mask, ["_wid", "label"]].groupby("_wid")["label"].max()

print(f"\nWindow stratification:")
for label_val, label_name in [(0, "normal"), (1, "attack")]:
    wids     = windows.index[windows == label_val].tolist()
    perm     = rng_split.permutation(len(wids))
    shuffled = [wids[i] for i in perm]
    n_test   = int(len(shuffled) * TEST_RATIO)
    test_m   = non_fault_mask & df_injected["_wid"].isin(set(shuffled[:n_test]))
    train_m  = non_fault_mask & df_injected["_wid"].isin(set(shuffled[n_test:]))
    df_injected.loc[test_m,  "split"] = "test"
    df_injected.loc[train_m, "split"] = "train"
    print(f"  {label_name:8s}: {len(wids):6,} windows → test={n_test:,}, train={len(wids)-n_test:,}")

df_injected = df_injected.drop(columns=["_wid"], errors="ignore")

print(f"\nFinal split assignment:")
for split in ["train", "test"]:
    n0 = ((df_injected["split"] == split) & (df_injected["label"] == 0)).sum()
    n1 = ((df_injected["split"] == split) & (df_injected["label"] == 1)).sum()
    n2 = ((df_injected["split"] == split) & (df_injected["label"] == 2)).sum()
    print(f"  {split:<6}: {n0+n1+n2:>9,} rows  (normal={n0:,}  attack={n1:,}  fault={n2:,})")

Fault event stratification:
  bias                          :  26 events → test=5, train=21
  drift                         :  40 events → test=8, train=32
  intermittent_dropout          :  55 events → test=11, train=44
  precision_degradation         :  15 events → test=3, train=12
  stuck_at                      :  17 events → test=3, train=14

Window stratification:
  normal  :  1,258 windows → test=251, train=1,007
  attack  :    258 windows → test=51, train=207

Final split assignment:
  train :   772,812 rows  (normal=613,680  attack=8,063  fault=151,069)
  test  :   181,585 rows  (normal=144,118  attack=1,914  fault=35,553)


## 9. Save

Writes the three-class dataset with train/test split assignments to **`data/wadi_faulted.parquet`**. Consumed by NB3 for feature engineering.

In [10]:
out_path = DATA_DIR / "wadi_faulted.parquet"
df_injected.to_parquet(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Shape: {df_injected.shape}")
print(f"Completed: {datetime.now()}")
print(f"\nNext: Run NB3, then NB4.")


Saved: data/wadi_faulted.parquet
Shape: (954399, 110)
Completed: 2026-04-24 14:59:06.634417

Next: Run NB3, then NB4.
